# Task 3: Text Preprocessing Pipeline (Hugging Face Transformers)

**Goal:** Preprocess text descriptions into tokenized and encoded representations
(embeddings) that a text-to-image model can use as input.

**Model used:** CLIP text encoder (`openai/clip-vit-base-patch32`) via Hugging
Face Transformers. CLIP is trained specifically so text and image embeddings
land in the same vector space — exactly what a text-to-image pipeline needs,
and it sets us up well for Task 5 (attention-based GAN) and Task 6 (full
pipeline).

**Input:** `flowers_dataset_processed.csv` (from Task 4 — filenames, labels,
class names, captions)

**Output:** A saved set of caption embeddings, reusable directly in Tasks 2,
5, and 6.

Run this on Colab (GPU optional but speeds things up — Runtime > Change
runtime type > T4 GPU).


In [2]:
!pip install -q transformers torch pandas


## 1. Upload Task 4's output

Upload `flowers_dataset_processed.csv` (the file saved at the end of the
Task 4 notebook) using the file browser on the left, or the cell below.


In [3]:
from google.colab import files

uploaded = files.upload()  # select flowers_dataset_processed.csv when prompted


Saving flowers_dataset_processed (1).csv to flowers_dataset_processed (1).csv


In [5]:
import pandas as pd

df = pd.read_csv("flowers_dataset_processed (1).csv")
print(f"Loaded {len(df)} rows")
df.head()


Loaded 8189 rows


,filename,label,class_name,caption,caption_length
0,image_00001.jpg,77,passion flower,a photo of a passion flower,6
1,image_00002.jpg,77,passion flower,a photo of a passion flower,6
2,image_00003.jpg,77,passion flower,a photo of a passion flower,6
3,image_00004.jpg,77,passion flower,a photo of a passion flower,6
4,image_00005.jpg,77,passion flower,a photo of a passion flower,6


## 2. Load the CLIP tokenizer and text encoder


In [6]:
import torch
from transformers import CLIPTokenizer, CLIPTextModel

MODEL_NAME = "openai/clip-vit-base-patch32"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME).to(device)
text_encoder.eval()

print("Tokenizer and text encoder loaded.")


Using device: cpu


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.pre_layrnorm.bias                                 | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.l

Tokenizer and text encoder loaded.


## 3. Tokenize a sample caption (sanity check)

Before running this on the whole dataset, let's see what tokenization
actually looks like on one caption.


In [7]:
sample_caption = df["caption"].iloc[0]
print(f"Caption: {sample_caption}")

tokens = tokenizer(sample_caption, padding="max_length", max_length=32,
                    truncation=True, return_tensors="pt")

print(f"Token IDs shape: {tokens['input_ids'].shape}")
print(f"Token IDs: {tokens['input_ids']}")

decoded = tokenizer.decode(tokens["input_ids"][0])
print(f"Decoded back: {decoded}")


Caption: a photo of a passion flower
Token IDs shape: torch.Size([1, 32])
Token IDs: tensor([[49406,   320,  1125,   539,   320,  5332,  4055, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407]])
Decoded back: <|startoftext|>a photo of a passion flower <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


## 4. Encode the sample caption into an embedding


In [8]:
with torch.no_grad():
    tokens = {k: v.to(device) for k, v in tokens.items()}
    output = text_encoder(**tokens)

# CLIP gives per-token embeddings (last_hidden_state) and a pooled sentence-level embedding
print(f"Per-token embeddings shape: {output.last_hidden_state.shape}")   # (1, seq_len, hidden_dim)
print(f"Pooled embedding shape: {output.pooler_output.shape}")            # (1, hidden_dim)


Per-token embeddings shape: torch.Size([1, 32, 512])
Pooled embedding shape: torch.Size([1, 512])


## 5. Batch-process the whole dataset

Now we tokenize and encode every caption in the dataset, in batches, and
keep the pooled embedding per caption (one vector per flower image — this
is what Tasks 2, 5, and 6 will condition on).


In [9]:
from tqdm import tqdm
import numpy as np

BATCH_SIZE = 64
all_embeddings = []

captions = df["caption"].tolist()

for i in tqdm(range(0, len(captions), BATCH_SIZE)):
    batch_captions = captions[i:i + BATCH_SIZE]
    tokens = tokenizer(batch_captions, padding="max_length", max_length=32,
                        truncation=True, return_tensors="pt")
    tokens = {k: v.to(device) for k, v in tokens.items()}

    with torch.no_grad():
        output = text_encoder(**tokens)

    all_embeddings.append(output.pooler_output.cpu().numpy())

embeddings = np.concatenate(all_embeddings, axis=0)
print(f"Final embeddings shape: {embeddings.shape}")  # (num_images, hidden_dim)


100%|██████████| 128/128 [07:30<00:00,  3.52s/it]

Final embeddings shape: (8189, 512)


## 6. Sanity-check the embeddings

Captions describing the same flower class should have embeddings closer to
each other than to unrelated classes (since our captions are template-based
per class, this should be a very clean signal).


In [10]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Compare two captions from the SAME class vs DIFFERENT classes
same_class_idxs = df[df["label"] == df["label"].iloc[0]].index[:2]
diff_class_idx = df[df["label"] != df["label"].iloc[0]].index[0]

sim_same = cosine_sim(embeddings[same_class_idxs[0]], embeddings[same_class_idxs[1]])
sim_diff = cosine_sim(embeddings[same_class_idxs[0]], embeddings[diff_class_idx])

print(f"Similarity within same class: {sim_same:.4f}")
print(f"Similarity across different classes: {sim_diff:.4f}")
print("(Same-class similarity should be higher, since captions are identical per class)")


Similarity within same class: 1.0000
Similarity across different classes: 0.5602
(Same-class similarity should be higher, since captions are identical per class)


## 7. Save embeddings for reuse in Tasks 2, 5, 6


In [11]:
np.save("caption_embeddings.npy", embeddings)

# Also save a mapping so we know which row/filename each embedding belongs to
df[["filename", "label", "class_name", "caption"]].to_csv("embeddings_index.csv", index=False)

print("Saved caption_embeddings.npy and embeddings_index.csv")
print(f"Embeddings shape: {embeddings.shape}")


Saved caption_embeddings.npy and embeddings_index.csv
Embeddings shape: (8189, 512)


In [12]:
from google.colab import files

files.download("caption_embeddings.npy")
files.download("embeddings_index.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Summary

Text descriptions from the Oxford-102 Flowers dataset were successfully tokenized and encoded using the pre-trained CLIP text encoder. The resulting sentence-level embeddings provide numerical representations of the flower descriptions that can be used as conditional information in later text-to-image generation experiments.

The following cell automatically reports the final preprocessing and embedding statistics after execution.


In [13]:
print("===== TASK 3 SUMMARY =====")
print("Text encoder used: CLIP (openai/clip-vit-base-patch32)")
print("Tokenizer max sequence length: 32 tokens")
print(f"Total captions processed: {len(df)}")
print(f"Embedding dimensionality: {embeddings.shape[1]}")
print(f"Same-class cosine similarity: {sim_same:.4f}")
print(f"Different-class cosine similarity: {sim_diff:.4f}")

print(
    "\nObservation: CLIP successfully converted the textual flower "
    "descriptions into fixed-dimensional numerical embeddings. Captions "
    "belonging to the same flower class are identical in this template-based "
    "dataset and therefore produce identical or highly similar embeddings. "
    "Different flower descriptions produce different embedding representations. "
    "These embeddings can be used as textual conditioning information for later "
    "text-to-image generation tasks."
)


===== TASK 3 SUMMARY =====
Text encoder used: CLIP (openai/clip-vit-base-patch32)
Tokenizer max sequence length: 32 tokens
Total captions processed: 8189
Embedding dimensionality: 512
Same-class cosine similarity: 1.0000
Different-class cosine similarity: 0.5602

Observation: CLIP successfully converted the textual flower descriptions into fixed-dimensional numerical embeddings. Captions belonging to the same flower class are identical in this template-based dataset and therefore produce identical or highly similar embeddings. Different flower descriptions produce different embedding representations. These embeddings can be used as textual conditioning information for later text-to-image generation tasks.
